# Task 1.6

#### Importing libraries

In [1]:
# Importing libraries
import pandas as pd
import re
from thefuzz import process
import numpy as np
import spacy
from spacy import displacy
import networkx as nx
import os
import matplotlib.pyplot as plt
import scipy

## Data cleanup

In [2]:
# loading country names from the Excel file
df_countries = pd.read_csv('countries_list_20th_century_1.5.csv')

In [3]:
# cleaning country list
df_countries['country_name'] = df_countries['country_name'].str.split(',').str[0].str.strip()

In [4]:
df_countries.head()

,Unnamed: 0,country_name
0,1,Afghanistan
1,2,Albania
2,3,Algeria
3,4,Andorra
4,5,Angola


In [5]:
# converting country names to list format
standard_countries = df_countries['country_name'].tolist()

In [6]:
# open 20th century txt
with open('20th_Century_content.txt', 'r', encoding='utf-8') as file:
    text = file.read()

In [7]:
# remove extra spaces, newlines, and special characters (modify regex as needed)
text = re.sub(r"\s+", " ", text).strip()  # replace multiple spaces/newlines with a single space

In [8]:
print(text)

From Wikipedia, the free encyclopedia The 20th century changed the world in unprecedented ways. The World Wars sparked tension between countries and led to the creation of atomic bombs, the Cold War led to the Space Race and the creation of space-based rockets, and the World Wide Web was created. These advancements have played a significant role in citizens' lives and shaped the 21st century into what it is today. Historic events in the 20th century[edit] World at the beginning of the century[edit] Main article: Edwardian era The new beginning of the 20th century marked significant changes. The 1900s saw the decade herald a series of inventions, including the automobile, airplane and radio broadcasting. 1914 saw the completion of the Panama Canal. The Scramble for Africa continued in the 1900s and resulted in wars and genocide across the continent. The atrocities in the Congo Free State shocked the civilized world. From 1914 to 1918, the First World War, and its aftermath, caused major

In [9]:
# extract potential country names (using regex for capitalized words)
words = re.findall(r'\b[A-Z][a-z]+\b(?:\s[A-Z][a-z]+)*', text)

In [10]:
# fuzzy match and replace names in the text
def standardize_country(name):
    match, score = process.extractOne(name, standard_countries)
    return match if score > 90 else name # 90% is the threshold

standardized_words = {word: standardize_country(word) for word in set(words)}

In [11]:
# replace occurrences in text
for incorrect, correct in standardized_words.items():
    text = text.replace(incorrect, correct)

In [12]:
# save cleaned text
with open('Cleaned_Wiki_2.txt', 'w', encoding='utf-8') as file:
    file.write(text)

In [13]:
print("Standardization Complete. Check 'Cleaned_Wiki_2.txt'.")

Standardization Complete. Check 'Cleaned_Wiki_2.txt'.


In [14]:
# check changes made
for incorrect, correct in standardized_words.items():
    if incorrect != correct:
        print(f"Replaced: {incorrect} -> {correct}")

Replaced: Latvian -> Latvia
Replaced: Indian -> India
Replaced: Russian -> Russia
Replaced: Australian -> Australia
Replaced: The Philippines -> Philippines
Replaced: Koreas -> Korea
Replaced: Estonian -> Estonia
Replaced: The United States -> United States
Replaced: The Netherlands -> Netherlands
Replaced: German -> Germany
Replaced: Lithuanian -> Lithuania


In [15]:
# check for other non standard names
remaining_countries = [word for word in words if word not in standard_countries]
print("Potential unmatched country names:", set(remaining_countries))

Potential unmatched country names: {'Nanjing', 'Italian', 'Final Victory', 'Human', 'Related Networks', 'Wake Island', 'Global', 'Leyte Gulf', 'World Scientific', 'Henson', 'No', 'West Nile', 'Panama Canal', 'Manchukuo', 'Immediately', 'Peck', 'Cerf', 'King', 'The Decade We Almost Stopped Climate Change', 'Royal Air Force', 'Hasic', 'Gandhi', 'International Law', 'Steve', 'Tim', 'Estrada', 'Their Secret World War', 'Online Interactive', 'Rivalries', 'Little Boy', 'June', 'Centers', 'Japanese Instrument', 'Can Help Further', 'Kennedy Presidential Library', 'Roosevelt And Stalin Planned To End The Second World War', 'Even Though He Did Not Land Inside His Spacecraft', 'Cambridge', 'Masters', 'The Apollo Program', 'Began', 'National Socialist Guerrilla Movement', 'Jehovah', 'National Archives Foundation', 'From World War', 'Archduke Franz Ferdinand The First World War', 'Central Intelligence Agency', 'General Douglas Mac', 'At', 'Skylab', 'Fenton', 'Sam', 'The Pacific War', 'Latvian', 'Fi

#### 4. some errors are there but not very important

In [16]:
# load spacy English module
NER = spacy.load('en_core_web_sm')

# Load the Wiki Text

In [17]:
book = NER(text)

In [18]:
# visualize identified entities
displacy.render(book[273:20000], style = "ent", jupyter = True)

# Named Entity List per Sentence

In [19]:
df_sentences = [] # empty shell to store results

# loop through sentences, get entity list for each sentence
for sent in book.sents:
    entity_list = [ent.text for ent in sent.ents]
    df_sentences.append({"sentence": sent, "entities": entity_list})
    
df_sentences = pd.DataFrame(df_sentences)

In [20]:
df_sentences.head(10)

,sentence,entities
0,"(From, Wikipedia, ,, the, free, encyclopedia, ...","[Wikipedia, The 20th century]"
1,"(The, World, Wars, sparked, tension, between, ...","[The World Wars, the Cold War, the Space Race,..."
2,"(These, advancements, have, played, a, signifi...","[the 21st century, today]"
3,"(Historic, events, in, the, 20th, century[edit...","[20th, the 20th century]"
4,"(The, 1900s, saw, the, decade, herald, a, seri...","[The 1900s, the decade]"
5,"(1914, saw, the, completion, of, the, Panama, ...","[1914, the Panama Canal]"
6,"(The, Scramble, for, Africa, continued, in, th...","[Scramble, Africa, the 1900s]"
7,"(The, atrocities, in, the, Congo, Free, State,...",[the Congo Free State]
8,"(From, 1914, to, 1918, ,, the, First, World, W...","[1914 to 1918, the First World War]"
9,"("", The, war, to, end, all, wars, "", :, World,...","[World War I, World War I Arrest, Sarajevo, Ar..."


# Filtering Countries from Book

In [21]:
# checking all country names
print(df_countries['country_name'].unique())  

['Afghanistan' 'Albania' 'Algeria' 'Andorra' 'Angola'
 'Antigua and Barbuda' 'Argentina' 'Armenia' 'Australia' 'Austria'
 'Azerbaijan' 'Bahamas' 'Bahrain' 'Bangladesh' 'Barbados' 'Belarus'
 'Belgium' 'Belize' 'Benin' 'Bhutan' 'Bolivia' 'Bosnia and Herzegovina'
 'Botswana' 'Brazil' 'Brunei' 'Bulgaria' 'Burkina Faso' 'Burundi'
 'Cambodia' 'Cameroon' 'Canada' 'Cape Verde' 'Central African Republic'
 'Chad' 'Chile' 'China' 'Colombia' 'Comoros' 'Congo' 'Costa Rica'
 'Croatia' 'Cuba' 'Cyprus' 'Czech Republic' 'Denmark' 'Djibouti'
 'Dominica' 'Dominican Republic' 'East Timor' 'Ecuador' 'Egypt'
 'El Salvador' 'Equatorial Guinea' 'Eritrea' 'Estonia' 'Eswatini'
 'Ethiopia' 'Fiji' 'Finland' 'France' 'Gabon' 'Gambia' 'Georgia' 'Germany'
 'Ghana' 'Greece' 'Grenada' 'Guatemala' 'Guinea' 'Bissau' 'Guyana' 'Haiti'
 'Honduras' 'Hungary' 'Iceland' 'India' 'Indonesia' 'Iran' 'Iraq'
 'Ireland' 'Israel' 'Italy' 'Ivory Coast' 'Jamaica' 'Japan' 'Jordan'
 'Kazakhstan' 'Kenya' 'Kiribati' 'Korea' 'Kuwait' 'Kyrg

In [22]:
# filtering out entities not of interest
def filter_entity(ent_list, df_countries):
    return [ent for ent in ent_list 
            if ent in list(df_countries['country_name'])]

In [23]:
# testing
filter_entity(["Germany", "Bahamas", "2"], df_countries)

['Germany', 'Bahamas']

In [24]:
df_countries_list = list(df_countries['country_name']) 

def filter_entity(ent_list, countries_list):
    return [ent for ent in ent_list if ent in countries_list]

df_sentences['country_entities'] = df_sentences['entities'].apply(lambda x: filter_entity(x, df_countries_list))

In [25]:
df_sentences.head(150)

,sentence,entities,country_entities
0,"(From, Wikipedia, ,, the, free, encyclopedia, ...","[Wikipedia, The 20th century]",[]
1,"(The, World, Wars, sparked, tension, between, ...","[The World Wars, the Cold War, the Space Race,...",[]
2,"(These, advancements, have, played, a, signifi...","[the 21st century, today]",[]
3,"(Historic, events, in, the, 20th, century[edit...","[20th, the 20th century]",[]
4,"(The, 1900s, saw, the, decade, herald, a, seri...","[The 1900s, the decade]",[]
...,...,...,...
145,"(At, this, time, ,, Hitler, introduced, the, V...","[Hitler, V-1, V-2, first]",[]
146,"(The, V-1, was, often, intercepted, by, air, p...","[V-1, V-2]",[]
147,"(However, ,, this, advance, came, too, late, i...",[],[]
148,"(The, Germanys, were, also, on, the, verge, of...","[Germanys, Germanys, Atlantic]",[]


In [26]:
unique_countries = set([country for sublist in df_sentences['country_entities'] for country in sublist])
print(unique_countries)

{'Germany', 'Italy', 'Egypt', 'Romania', 'Albania', 'Austria', 'Solomon Islands', 'Lithuania', 'Kenya', 'Netherlands', 'Korea', 'Canada', 'Finland', 'Mongolia', 'Vietnam', 'Bulgaria', 'Spain', 'Thailand', 'Latvia', 'Norway', 'Afghanistan', 'Philippines', 'Belarus', 'Seychelles', 'Denmark', 'Ghana', 'Japan', 'United States', 'Papua New Guinea', 'Hungary', 'Belgium', 'Iraq', 'Morocco', 'Algeria', 'Cuba', 'Cambodia', 'Ukraine', 'Greece', 'Rwanda', 'Sweden', 'Israel', 'Estonia', 'Poland', 'Pakistan', 'Laos', 'Lebanon', 'South Africa', 'Russia', 'Singapore', 'India', 'Iran', 'Australia', 'Libya', 'France', 'Luxembourg', 'China'}


In [27]:
# filter out sentences that don't have any character entities
df_sentences_filtered = df_sentences[df_sentences['country_entities'].map(len) > 0]

In [28]:
df_sentences_filtered.tail(10)

,sentence,entities,country_entities
1101,"("", The, forgotten, violence, that, helped, In...",[India],[India]
1106,"("", India, Independence, Day, :, everything, y...","[India Independence Day, Partition between Ind...",[Pakistan]
1115,"(^, "", Philippines, ,, 1898–1946, |, US, House...","[Philippines, 1898–1946, US House of Represent...",[Philippines]
1145,"("", Colonial, Cartographies, ,, Postcolonial, ...","[Colonial Cartographies, Postcolonial Borders,...",[Afghanistan]
1182,"(The, Moldovans, :, Romania, ,, Russia, ,, and...","[Moldovans, Romania, Russia, the Politics of C...","[Romania, Russia]"
1245,"("", Selling, ', Operation, Passage, to, Freedo...","[Thomas Dooley, the Religious Overtones of Ear...",[Vietnam]
1277,"("", Stuck, in, Endless, Preliminaries, :, Viet...","[Stuck in Endless Preliminaries, Vietnam, the ...",[Vietnam]
1548,"("", Anti, -, American, Behavior, in, the, Midd...","[Anti-American Behavior, the Middle East, a Fi...",[Lebanon]
1554,"(The, Rise, of, China, and, India, :, A, New, ...","[India, New Asian]",[India]
1555,"(Singapore, :, World, Scientific, ., doi:10.11...","[Singapore, World Scientific]",[Singapore]


# Create Relationships

In [29]:
# defining relationships 
# window size = 5 : defines how many sentences will be looked at simultaneously 
relationships = [] # create an empty list

for i in range(df_sentences_filtered.index[-1]):
    end_i = min(i+5, df_sentences_filtered.index[-1])
    country_list = sum((df_sentences_filtered.loc[i: end_i].country_entities), [])
    
    # remove duplicated characters that are next to each other
    country_unique = [country_list[i] for i in range(len(country_list)) 
                   if (i==0) or country_list[i] != country_list[i-1]]
    
    if len(country_unique) > 1:
        for idx, a in enumerate(country_unique[:-1]):
            b = country_unique[idx + 1]
            relationships.append({"source": a, "target": b})

In [30]:
df_relationship = pd.DataFrame(relationships)

In [31]:
df_relationship

,source,target
0,France,Austria
1,France,Austria
2,Austria,Russia
3,France,Austria
4,Austria,Russia
...,...,...
635,India,Singapore
636,India,Singapore
637,India,Singapore
638,India,Singapore


In [32]:
# sort cases with a->b and b->a
df_relationship = pd.DataFrame(np.sort(df_relationship.values, axis = 1), columns = df_relationship.columns)
df_relationship.head(5)

,source,target
0,Austria,France
1,Austria,France
2,Austria,Russia
3,Austria,France
4,Austria,Russia


In [33]:
# summarize interactions
df_relationship["value"] = 1
df_relationship = df_relationship.groupby(["source","target"], sort=False, as_index=False).sum()

In [34]:
df_relationship

,source,target,value
0,Austria,France,6
1,Austria,Russia,5
2,Germany,Italy,22
3,Austria,Germany,9
4,Germany,Spain,3
...,...,...,...
85,Greece,Italy,3
86,Italy,Japan,6
87,Australia,Philippines,3
88,Romania,Russia,6


In [35]:
sorted_df = df_relationship.sort_values(by='value', ascending=False)
sorted_df

,source,target,value
35,China,Japan,29
2,Germany,Italy,22
5,France,Poland,21
47,China,India,17
19,France,Germany,16
...,...,...,...
57,India,Papua New Guinea,3
26,Belarus,Libya,2
41,Australia,Solomon Islands,2
81,Cambodia,United States,2


In [36]:
df_relationship.to_csv('country_relationship.csv')